# Affective & Readability Features (5 features)

Extracts five  features per document:

| Code | Column name | What it measures |
|---|---|---|
| `WRDVALm`  | `word_valence_mean`    | Mean Warriner valence over all alphabetic tokens (1 = highly negative, 5 = neutral, 9 = highly positive) |
| `WRDVALmc` | `word_valence_content` | Mean Warriner valence over content words only (NOUN/VERB/ADJ/ADV) |
| `WRDTRNm`  | `word_arousal_mean`    | Mean Warriner arousal over all alphabetic tokens (1 = calm, 9 = excited) |
| `WRDTRNmc` | `word_arousal_content` | Mean Warriner arousal over content words only |
| `READNDC`  | `dale_chall_readability` | New Dale–Chall readability score (uses Chall & Dale's 3,000-word familiar-word list) |



## 1 · Install

In [1]:
!pip install pandas spacy textstat

## 2 · Imports & NLP backbone

In [2]:
import os, sys, json, hashlib, platform
from pathlib import Path
from time import time

import numpy as np
import pandas as pd
import spacy
import textstat

# Lightweight pipeline — we only need the tagger + lemmatiser (for the content-word
# filter and lemma lookup in Warriner). Parser, NER, and sentencisation are NOT used
# by any of the five features (Dale–Chall does its own sentence splitting inside
# textstat), so we disable them for ~3× speedup.
nlp = spacy.load('en_core_web_sm', disable=['parser', 'ner'])
print(f'spaCy pipeline: {nlp.pipe_names}')

# Reproducibility — captured for the snapshot at the end
ENV = {
    'python':   sys.version.split()[0],
    'platform': platform.platform(),
    'spacy':    spacy.__version__,
    'numpy':    np.__version__,
    'pandas':   pd.__version__,
    'textstat': textstat.__version__,
}
print(json.dumps(ENV, indent=2))

spaCy pipeline: ['tok2vec', 'tagger', 'attribute_ruler', 'lemmatizer']
{
  "python": "3.12.13",
  "platform": "Linux-6.6.122+-x86_64-with-glibc2.35",
  "spacy": "3.8.14",
  "numpy": "1.26.4",
  "pandas": "2.2.2",
  "textstat": [
    0,
    7,
    2
  ]
}


## 3 · Configuration

In [3]:
# ── Input ────────────────────────────────────────────────────────────────────
DATA_PATH_CANDIDATES = [
   '/content/english_final_truncated.xlsx'
]
DATA_PATH = next((p for p in DATA_PATH_CANDIDATES if os.path.exists(p)), None)
if DATA_PATH is None:
    raise FileNotFoundError(f'No data file found in {DATA_PATH_CANDIDATES}')
print(f'DATA_PATH = {DATA_PATH}')

TEXT_COL  = 'content'
INDEX_COL = 'index'
GRADE_COL = 'grade'      # used only for monotonicity sanity check; skipped if missing

# ── Output ───────────────────────────────────────────────────────────────────
OUTPUT_SOLO   = 'affective_readability_5feat.xlsx'
OUTPUT_MERGED = 'english_only_feature_full_PLUS5.xlsx'
SNAPSHOT_JSON = 'affective_readability_snapshot.json'

# ── Reproducibility ──────────────────────────────────────────────────────────
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

DATA_PATH = /content/english_final_truncated.xlsx


## 4 · Load corpus

In [4]:
df = pd.read_excel(DATA_PATH)
df.columns = [c.strip() for c in df.columns]
assert TEXT_COL  in df.columns, f"Required column '{TEXT_COL}' not found"
assert INDEX_COL in df.columns, f"Required column '{INDEX_COL}' not found"

print(f'Loaded {len(df)} essays · {df.shape[1]} columns')
if GRADE_COL in df.columns:
    print(f'Grade distribution:')
    print(df[GRADE_COL].value_counts().sort_index().to_string())
else:
    print(f"⚠ No '{GRADE_COL}' column — grade-correlation sanity check will skip.")

Loaded 172 essays · 635 columns
Grade distribution:
grade
2    26
3    19
4    21
5    27
6    27
7    26
8    26


## 5 · Load Warriner affective norms

**Source:** Warriner et al. (2013), *Behavior Research Methods* 45(4):1191–1207.
Scale: 1–9 (1 = highly negative / calm, 9 = highly positive / excited). We use the mean ratings (`V.Mean.Sum`, `A.Mean.Sum`).

We try the canonical CRR URL first, fall back to a GitHub mirror used by mainstream NLP toolkits (XANEW project). The downloaded file is MD5-hashed and the hash is recorded in the reproducibility snapshot.

In [5]:
WARRINER_URLS = [
    'http://crr.ugent.be/papers/Ratings_Warriner_et_al.csv',                              # canonical (Crystallizing Reading Resources)
    'https://raw.githubusercontent.com/JULIELab/XANEW/master/Ratings_Warriner_et_al.csv', # mirror
]
WARRINER_LOCAL = 'Ratings_Warriner_et_al.csv'

def load_warriner():
    # 1 · prefer local cache for full determinism on re-runs
    if os.path.exists(WARRINER_LOCAL):
        print(f'Using cached {WARRINER_LOCAL}')
        return pd.read_csv(WARRINER_LOCAL), f'file://{os.path.abspath(WARRINER_LOCAL)}'
    # 2 · network fallbacks
    last_err = None
    for url in WARRINER_URLS:
        try:
            print(f'Trying {url} ...')
            df_w = pd.read_csv(url)
            df_w.to_csv(WARRINER_LOCAL, index=False)   # cache for next run
            print(f'  ✓ Loaded {len(df_w):,} rows (cached to {WARRINER_LOCAL})')
            return df_w, url
        except Exception as e:
            print(f'  ✗ failed: {type(e).__name__}: {e}')
            last_err = e
    raise RuntimeError(f'Could not download Warriner norms: {last_err}')

warriner_df, WARRINER_SOURCE_URL = load_warriner()

# Robust column resolution (different mirrors use slightly different headers)
val_col  = next((c for c in warriner_df.columns if 'V.Mean' in c or c.lower() == 'valence'), None)
aro_col  = next((c for c in warriner_df.columns if 'A.Mean' in c or c.lower() == 'arousal'), None)
word_col = next((c for c in warriner_df.columns if c.lower() in ('word', 'lemma')), None)
assert val_col and aro_col and word_col, f'Cannot resolve V/A/word columns in {list(warriner_df.columns)}'
print(f'\nResolved columns: word={word_col!r}  valence={val_col!r}  arousal={aro_col!r}')

# Build lowercased lookup dicts. If the file has duplicate lemmas (rare),
# we average — guarantees determinism regardless of row order.
warriner_df['_w'] = warriner_df[word_col].astype(str).str.lower().str.strip()
VAL_DICT = warriner_df.groupby('_w')[val_col].mean().astype(float).to_dict()
ARO_DICT = warriner_df.groupby('_w')[aro_col].mean().astype(float).to_dict()

# Range check on the norms themselves — should be ~1-9
_vals = np.array(list(VAL_DICT.values()))
_aros = np.array(list(ARO_DICT.values()))
assert 1.0 <= _vals.min() and _vals.max() <= 9.0, f'Valence out of [1,9]: [{_vals.min():.2f}, {_vals.max():.2f}]'
assert 1.0 <= _aros.min() and _aros.max() <= 9.0, f'Arousal out of [1,9]: [{_aros.min():.2f}, {_aros.max():.2f}]'

print(f'\n✓ Built VAL_DICT ({len(VAL_DICT):,} entries)  range=[{_vals.min():.2f}, {_vals.max():.2f}]  mean={_vals.mean():.2f}')
print(f'✓ Built ARO_DICT ({len(ARO_DICT):,} entries)  range=[{_aros.min():.2f}, {_aros.max():.2f}]  mean={_aros.mean():.2f}')

print(f'\nProbe lookups (sanity):')
for w in ['happy', 'sad', 'calm', 'terrified', 'neutral', 'love', 'hate', 'murder', 'baby']:
    v, a = VAL_DICT.get(w), ARO_DICT.get(w)
    print(f'  {w:<12}  val={v:.2f}  aro={a:.2f}' if v is not None else f'  {w:<12}  (not in Warriner)')

Trying http://crr.ugent.be/papers/Ratings_Warriner_et_al.csv ...
  ✗ failed: HTTPError: HTTP Error 404: Not Found
Trying https://raw.githubusercontent.com/JULIELab/XANEW/master/Ratings_Warriner_et_al.csv ...
  ✓ Loaded 13,915 rows (cached to Ratings_Warriner_et_al.csv)

Resolved columns: word='Word'  valence='V.Mean.Sum'  arousal='A.Mean.Sum'

✓ Built VAL_DICT (13,905 entries)  range=[1.26, 8.53]  mean=5.06
✓ Built ARO_DICT (13,905 entries)  range=[1.60, 7.79]  mean=4.21

Probe lookups (sanity):
  happy         val=8.47  aro=6.05
  sad           val=2.10  aro=3.49
  calm          val=6.89  aro=1.67
  terrified     val=2.51  aro=6.10
  neutral       val=5.50  aro=3.45
  love          val=8.00  aro=5.36
  hate          val=1.96  aro=6.26
  murder        val=1.48  aro=6.24
  baby          val=6.67  aro=4.97


## 6 · Per-document feature extractor

All five features are computed in a single pass per spaCy `Doc`.

### Definitions

Let `T` be the multiset of alphabetic tokens in the document (`token.is_alpha == True`) and `C ⊆ T` the content tokens with POS ∈ {NOUN, VERB, ADJ, ADV}. For each token we look up its lowercased lemma (`token.lemma_.lower()`) in the Warriner table.

- **`WRDVALm`** = mean of `VAL_DICT[lemma]` over `{t ∈ T : lemma(t) ∈ VAL_DICT}`. `NaN` if the set is empty.
- **`WRDVALmc`** = same as above but restricted to `C`.
- **`WRDTRNm`**  = same as `WRDVALm` but using `ARO_DICT`.
- **`WRDTRNmc`** = same as `WRDVALmc` but using `ARO_DICT`.
- **`READNDC`** = `textstat.dale_chall_readability_score(raw_text)`. Computed on the raw text (not the tokenised version) because Dale–Chall depends on its own sentence/word tokenisation and the 3,000-word familiar list. `NaN` if textstat raises (e.g. empty/very short text).

### Defensibility notes (for reviewer questions)

1. **Lemma vs surface form.** We use lemmas because Warriner *et al.* explicitly elicited ratings for lemmas. Inflected forms (`loved`, `loving`, `loves`) inherit the rating of their lemma (`love`). This matches the original norm collection protocol and prior work using these norms.
2. **AUX exclusion.** spaCy tags auxiliaries as `AUX`, not `VERB`, so `is/have/do` (as auxiliaries) are excluded from content words but their main-verb readings (`I *have* a book`) keep `AUX` and are excluded — by design.
3. **Coverage transparency.** A `_warriner_coverage` column (Warriner-hits / alphabetic-tokens) is tracked internally for QA. It's reported in §8 but **not** saved to the feature file, to keep the deliverable to exactly five columns.

In [6]:
CONTENT_POS = {'NOUN', 'VERB', 'ADJ', 'ADV'}

# Output column order — fixed for reproducibility
OUTPUT_COLUMNS = [
    'WRDVALm',    # word_valence_mean
    'WRDVALmc',   # word_valence_content
    'WRDTRNm',    # word_arousal_mean
    'WRDTRNmc',   # word_arousal_content
    'READNDC',    # dale_chall_readability
]

# Friendly names for plotting / reporting (not used as column names)
CODE_TO_NAME = {
    'WRDVALm':  'word_valence_mean',
    'WRDVALmc': 'word_valence_content',
    'WRDTRNm':  'word_arousal_mean',
    'WRDTRNmc': 'word_arousal_content',
    'READNDC':  'dale_chall_readability',
}


def _mean_norm(tokens, lookup):
    """Mean of `lookup[lemma]` over tokens whose lemma is in `lookup`.
    Returns (mean, n_hits, n_eligible). NaN mean when n_hits == 0."""
    eligible = len(tokens)
    vals = [lookup[t.lemma_.lower()] for t in tokens if t.lemma_.lower() in lookup]
    if not vals:
        return np.nan, 0, eligible
    return float(np.mean(vals)), len(vals), eligible


def extract_features(doc, raw_text):
    """Return dict of the 5 features plus internal coverage stats for one essay."""
    alpha_tokens = [t for t in doc if t.is_alpha]
    n_alpha = len(alpha_tokens)
    if n_alpha == 0:
        return {k: np.nan for k in OUTPUT_COLUMNS} | {
            '_warriner_cov_all': np.nan,
            '_warriner_cov_content': np.nan,
            '_n_alpha': 0,
            '_n_content': 0,
        }

    content_tokens = [t for t in alpha_tokens if t.pos_ in CONTENT_POS]

    val_all, hits_val_all, _   = _mean_norm(alpha_tokens,   VAL_DICT)
    val_con, hits_val_con, _   = _mean_norm(content_tokens, VAL_DICT)
    aro_all, hits_aro_all, _   = _mean_norm(alpha_tokens,   ARO_DICT)
    aro_con, hits_aro_con, _   = _mean_norm(content_tokens, ARO_DICT)

    # Dale–Chall on the *raw* text (textstat does its own sentence splitting)
    try:
        rdc = float(textstat.dale_chall_readability_score(raw_text))
    except Exception:
        rdc = np.nan

    return {
        'WRDVALm':              val_all,
        'WRDVALmc':             val_con,
        'WRDTRNm':              aro_all,
        'WRDTRNmc':             aro_con,
        'READNDC':              rdc,
        # internal QA only — dropped before save
        '_warriner_cov_all':     hits_val_all / n_alpha if n_alpha else 0.0,
        '_warriner_cov_content': (hits_val_con / len(content_tokens)) if content_tokens else 0.0,
        '_n_alpha':              n_alpha,
        '_n_content':            len(content_tokens),
    }

print(f'✓ Feature extractor defined. Will output {len(OUTPUT_COLUMNS)} columns: {OUTPUT_COLUMNS}')

✓ Feature extractor defined. Will output 5 columns: ['WRDVALm', 'WRDVALmc', 'WRDTRNm', 'WRDTRNmc', 'READNDC']


## 7 · Reference-text validation (face validity on known inputs)

Before running on the corpus we extract features on hand-crafted reference texts whose expected affective profile is unambiguous. This is the cheapest possible regression test — if any of these fail, the pipeline is broken regardless of how clean the corpus output looks.

Expected (from Warriner means: positive words ≈ 7.0+, negative words ≈ 2.5–3.5, calm words ≈ 3.0, intense words ≈ 6.0+):

| Reference text | Expected pattern |
|---|---|
| Positive / joyful | high valence (≥ 6.5), moderate arousal |
| Negative / fearful | low valence (≤ 4.5), high arousal (≥ 5.0) |
| Neutral / mundane | mid valence (≈ 5–6), low arousal (≤ 5.0) |
| Children's reader | low Dale–Chall (< 6.0) |
| Academic / formal | high Dale–Chall (≥ 8.0) |

In [7]:
REFERENCE_TEXTS = {
    'positive_joyful': (
        "My beautiful daughter laughed with joy when she saw the rainbow. "
        "We had a wonderful time at the park, playing games and sharing love. "
        "Her happy smile filled my heart with delight and warm gratitude."
    ),
    'negative_fearful': (
        "The terrified victim screamed as the murderer attacked with violent fury. "
        "Blood and terror filled the dark room. He suffered horribly, helpless against "
        "the cruel torture and inevitable death that awaited him."
    ),
    'neutral_mundane': (
        "The committee filed the report on Tuesday. The document contains tables and "
        "footnotes describing the inventory of office supplies. The chair is in the corner "
        "of the room next to the window."
    ),
    'childrens_reader': (
        "The little cat sat on the mat. The cat was small and grey. It liked to play with "
        "a red ball. The boy came home. He saw the cat. He gave the cat some milk. The cat "
        "was happy."
    ),
    'academic_formal': (
        "The phenomenological investigation of consciousness presupposes an epistemological "
        "framework wherein intentionality constitutes the irreducible substrate of cognitive "
        "experience, thereby necessitating a hermeneutic reconceptualisation of the "
        "subject–object dichotomy that has hitherto dominated analytic discourse."
    ),
}

EXPECTED = {
    'positive_joyful':  {'WRDVALm':  ('>=', 6.3), 'WRDVALmc': ('>=', 6.3)},
    'negative_fearful': {'WRDVALm':  ('<=', 4.7), 'WRDVALmc': ('<=', 4.7),
                         'WRDTRNm':  ('>=', 4.5), 'WRDTRNmc': ('>=', 4.5)},
    'neutral_mundane':  {'WRDTRNmc': ('<=', 5.2)},
    'childrens_reader': {'READNDC':  ('<=', 6.5)},
    'academic_formal':  {'READNDC':  ('>=', 8.0)},
}

print(f'{"text":<20} {"WRDVALm":>9} {"WRDVALmc":>10} {"WRDTRNm":>9} {"WRDTRNmc":>10} {"READNDC":>9}  expected-checks')
print('─' * 110)
ref_failures = []
for name, txt in REFERENCE_TEXTS.items():
    f = extract_features(nlp(txt), txt)
    checks = []
    for feat, (op, thr) in EXPECTED.get(name, {}).items():
        val = f[feat]
        ok = (val >= thr) if op == '>=' else (val <= thr)
        checks.append(f'{feat}{op}{thr}:{"✓" if ok else "✗"}')
        if not ok:
            ref_failures.append(f'{name} {feat}={val:.2f} fails {op}{thr}')
    row = f'{name:<20} {f["WRDVALm"]:>9.2f} {f["WRDVALmc"]:>10.2f} {f["WRDTRNm"]:>9.2f} {f["WRDTRNmc"]:>10.2f} {f["READNDC"]:>9.2f}  {" ".join(checks)}'
    print(row)

if ref_failures:
    print('\n⚠ Reference-text expectations FAILED:')
    for f in ref_failures:
        print(f'   {f}')
    print('   Investigate the pipeline before running on the corpus.')
else:
    print('\n✓ All reference-text expectations pass.')

text                   WRDVALm   WRDVALmc   WRDTRNm   WRDTRNmc   READNDC  expected-checks
──────────────────────────────────────────────────────────────────────────────────────────────────────────────
positive_joyful           7.09       7.16      4.55       4.61      6.92  WRDVALm>=6.3:✓ WRDVALmc>=6.3:✓
negative_fearful          3.09       3.09      5.23       5.23     11.57  WRDVALm<=4.7:✓ WRDVALmc<=4.7:✓ WRDTRNm>=4.5:✓ WRDTRNmc>=4.5:✓
neutral_mundane           5.40       5.35      3.22       3.20      8.61  WRDTRNmc<=5.2:✓
childrens_reader          6.57       6.61      4.00       4.06      4.72  READNDC<=6.5:✓
academic_formal           5.27       5.21      4.25       4.33     16.28  READNDC>=8.0:✓

✓ All reference-text expectations pass.


## 8 · Smoke test on first corpus essay

In [8]:
sample_text  = str(df[TEXT_COL].iloc[0])
sample_doc   = nlp(sample_text)
sample_feats = extract_features(sample_doc, sample_text)

print('Essay 0 (first 200 chars):')
print(sample_text[:200], '...\n')
print('Extracted features:')
for k in OUTPUT_COLUMNS:
    v = sample_feats[k]
    print(f'  {k:<10s} ({CODE_TO_NAME[k]:<24s}) : {v:.4f}' if not (isinstance(v, float) and np.isnan(v)) else f'  {k:<10s} : NaN')
print(f'\nInternal QA:')
print(f'  alphabetic tokens : {sample_feats["_n_alpha"]}')
print(f'  content tokens    : {sample_feats["_n_content"]}')
print(f'  Warriner cov (all)     : {sample_feats["_warriner_cov_all"]*100:.1f}%')
print(f'  Warriner cov (content) : {sample_feats["_warriner_cov_content"]*100:.1f}%')

Essay 0 (first 200 chars):
Once, there was a little girl named Tara. She lived with her mother. They were very poor. One day she went to a forest. There she met an old woman. The old woman gave her a pot. She said, "This is a m ...

Extracted features:
  WRDVALm    (word_valence_mean       ) : 6.0442
  WRDVALmc   (word_valence_content    ) : 6.0017
  WRDTRNm    (word_arousal_mean       ) : 4.0864
  WRDTRNmc   (word_arousal_content    ) : 4.2133
  READNDC    (dale_chall_readability  ) : 1.1300

Internal QA:
  alphabetic tokens : 252
  content tokens    : 117
  Warriner cov (all)     : 54.8%
  Warriner cov (content) : 87.2%


## 9 · Run on the full corpus

In [9]:
texts   = df[TEXT_COL].astype(str).tolist()
indices = df[INDEX_COL].tolist()

results = []
t0 = time()
# nlp.pipe is the fastest spaCy idiom; batch_size of 32 is a good default for short essays
for i, doc in enumerate(nlp.pipe(texts, batch_size=32, n_process=1)):
    feats = extract_features(doc, texts[i])
    feats[INDEX_COL] = indices[i]
    results.append(feats)
    if (i + 1) % 50 == 0:
        print(f'  processed {i+1}/{len(texts)}  ({time()-t0:.1f}s elapsed)')

print(f'\n✓ Done — {len(results)} essays in {time()-t0:.1f}s ({len(results)/(time()-t0):.1f} docs/s)')

out_df = pd.DataFrame(results)
# Reorder: index first, then features in fixed order, internal QA cols last
qa_cols = ['_warriner_cov_all', '_warriner_cov_content', '_n_alpha', '_n_content']
out_df = out_df[[INDEX_COL] + OUTPUT_COLUMNS + qa_cols]
out_df.head()

  processed 50/172  (2.5s elapsed)
  processed 100/172  (8.6s elapsed)
  processed 150/172  (12.6s elapsed)

✓ Done — 172 essays in 13.3s (13.0 docs/s)


,index,WRDVALm,WRDVALmc,WRDTRNm,WRDTRNmc,READNDC,_warriner_cov_all,_warriner_cov_content,_n_alpha,_n_content
0,1,6.044203,6.001667,4.086449,4.213333,1.13,0.547619,0.871795,252,117
1,2,5.858986,5.844394,3.916812,3.938939,5.90,0.479167,0.916667,144,72
2,3,6.098100,6.128726,3.889300,4.023057,5.18,0.467290,0.887006,428,177
3,4,5.933235,5.903793,3.884706,3.938621,7.43,0.507463,0.935484,67,31
4,5,6.160938,6.112500,3.334375,3.280833,4.84,0.477612,1.000000,67,24


In [16]:
# Save only the 5 output features (drop internal QA columns)
out_df_save = out_df[[INDEX_COL] + OUTPUT_COLUMNS].copy()

# 1 · Standalone file
out_df_save.to_excel(OUTPUT_SOLO, index=False)
print(f'✓ Saved {OUTPUT_SOLO}   shape={out_df_save.shape}')

# 2 · Merged with original corpus
df_orig = pd.read_excel(DATA_PATH)
df_orig.columns = [c.strip() for c in df_orig.columns]
df_merged = df_orig.merge(out_df_save, on=INDEX_COL, how='left')
assert len(df_merged) == len(df_orig), 'Merge changed row count!'
df_merged.to_excel(OUTPUT_MERGED, index=False)
print(f'✓ Saved {OUTPUT_MERGED}   shape={df_merged.shape}')
print(f'   Added {df_merged.shape[1] - df_orig.shape[1]} new columns: {OUTPUT_COLUMNS}')

✓ Saved affective_readability_5feat.xlsx   shape=(172, 6)
✓ Saved english_only_feature_full_PLUS5.xlsx   shape=(172, 640)
   Added 5 new columns: ['WRDVALm', 'WRDVALmc', 'WRDTRNm', 'WRDTRNmc', 'READNDC']
